# 応用編11

V3.6新機能:アクセス権限callable_to(呼び出し許可先リスト）

このノートブックでは、スマートコントラクト・バージョン3.6から利用できるcallable_toについて、使い方を説明します。 


## callable_toの概要

callable_toは、ユーザまたはコントラクトに対して設定されるアクセス権で、コントラクトの呼び出しを許可する機能を持ちます。  
ユーザAがコントラクトCを呼び出すには、ユーザAのcallable_toにコントラクトCが含まれ、コントラクトCのaccessible_toにユーザAが含まれる必要があります。  
callable_toのデフォルト値はall(すべて呼び出し可能)です。  

## 準備

設定やライブラリを読み込みます。  

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。

In [2]:
var wfstr = await loadWallet(`wallet-user0.json`);
var wallet0 = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
var uid0 = (await rpc.call(wallet0, 'c1query', { type: 'search', key: 'me' })).value[0].id;

In [3]:
var cid = '_eval@' + domain;

## callable_toで許可されないコントラクトを呼び出すとエラーとなる

ユーザuid0のcallable_toを空にします。

In [4]:
var resp = await rpc.call(adminWallet, 'c1update', { id: uid0, prop: 'callable_to', value: [ ] });
console.log(resp);

{
  txno: 703045,
  txid: 'xFmB26wJLM37XMwpHKSiB8dxADS8SrP9pue4DCdKNMhp9',
  status: 'ok',
  value: null
}


ユーザuid0のウォレットを使って、コントラクトcidを呼び出すとパーミッション・エラーになります。

In [5]:
var resp = await rpc.call(wallet0, cid);
console.log(resp);

{
  txno: 703046,
  txid: 'xaagX3jwx9hLJmjepbikJARrjeW5DkRS8dCeLJHxJtf7p',
  status: 'denied',
  value: 'callable permission'
}


## callable_toで許可されたコントラクトを呼び出す

ユーザuid0のcallable_toに、コントラクトcidを設定します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1update', { id: uid0, prop: 'callable_to', value: [ cid ] });
console.log(resp);

{
  txno: 703047,
  txid: 'xJip8TVzGTawAefbYced5eJLdLWFVHLSPndS5467SgjV3',
  status: 'ok',
  value: null
}


ユーザuid0はコントラクトcidの呼び出しに成功します。

In [7]:
var resp = await rpc.call(wallet0, cid);
console.log(resp);

{
  txno: 703048,
  txid: 'xAE43KC7pWW7wEMvRrbACv86BrGGNSNTFScjHpmUY67F2',
  status: 'ok',
  value: null
}


## クリーンアップ

ユーザuid0のcallable_toをallに戻しておきます。

In [8]:
var resp = await rpc.call(adminWallet, 'c1update', { id: uid0, prop: 'callable_to', value: [ 'all' ] });
console.log(resp);

{
  txno: 703049,
  txid: 'xxxcmwj2RTKe9BpugNxdRNy5cswfFKdFQgYsnhSGSdNkHB',
  status: 'ok',
  value: null
}
